In [ ]:
!pip -q install -U pip

# Стабильная связка под тренинг-скрипты
!pip -q install \
  diffusers==0.36.0 \
  transformers==4.57.3 \
  accelerate==1.12.0 \
  peft==0.18.1 \
  bitsandbytes==0.49.1 \
  safetensors \
  datasets \
  ftfy \
  tensorboard \
  sentencepiece \
  protobuf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.5 MB/s eta 0:00:00


In [ ]:
import os
import torch

import diffusers
import transformers
import accelerate
import peft

print("torch:", torch.__version__)
print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)


torch: 2.9.0+cu126
diffusers: 0.36.0
transformers: 4.57.3
accelerate: 1.12.0
peft: 0.18.1


In [ ]:
from PIL import Image
from glob import glob


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
import os, time, json, random
import torch

# ========== БАЗОВАЯ МОДЕЛЬ ==========
BASE_MODEL = "runwayml/stable-diffusion-v1-5"

# ========== ТВОЙ КОНЦЕПТ ==========
# редкий токен — чтобы в базе почти наверняка не было такого
INSTANCE_TOKEN = "sksfruitface"
CLASS_WORD = "fruit"

# основной промпт, который ты будешь использовать для тренинга/теста
INSTANCE_PROMPT = f"a photo of {INSTANCE_TOKEN} {CLASS_WORD} with a cute human face"
CLASS_PROMPT    = f"a photo of {CLASS_WORD}"

# ========== ПАПКИ ==========
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
OUT_ROOT = f"/content/drive/MyDrive/fruitface_project_{RUN_ID}"
os.makedirs(OUT_ROOT, exist_ok=True)

RAW_INSTANCE_DIR = "/content/fruitface/instance"       # тут твои 10 исходных картинок
INSTANCE_512_DIR = "/content/fruitface/instance_512"   # сюда сделаем 512x512

print("OUT_ROOT:", OUT_ROOT)
print("RAW_INSTANCE_DIR:", RAW_INSTANCE_DIR)
print("INSTANCE_512_DIR:", INSTANCE_512_DIR)

# ========== ТЕСТ ПРОМПТЫ ДЛЯ СРАВНЕНИЙ ==========
TEST_PROMPTS = [
    f"a photo of {INSTANCE_TOKEN} fruit with a cute human face, studio lighting, high detail",
    f"{INSTANCE_TOKEN} banana with a cute human face, macro photo, shallow depth of field",
    f"{INSTANCE_TOKEN} apple with a cute human face, cinematic lighting, 50mm",
    f"a bowl of {INSTANCE_TOKEN} fruits with cute human faces, professional food photography",
    f"{INSTANCE_TOKEN} orange with a cute human face, white background product photo",
]
NEG_PROMPT = "blurry, low quality, worst quality, deformed, bad anatomy"

# фиксируем сид для честного сравнения
GLOBAL_SEED = 123
random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

# сохраним метаданные для рапорта
meta = {
    "base_model": BASE_MODEL,
    "instance_token": INSTANCE_TOKEN,
    "class_word": CLASS_WORD,
    "instance_prompt": INSTANCE_PROMPT,
    "class_prompt": CLASS_PROMPT,
    "seed": GLOBAL_SEED,
}
with open(os.path.join(OUT_ROOT, "project_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta ->", os.path.join(OUT_ROOT, "project_meta.json"))


OUT_ROOT: /content/drive/MyDrive/fruitface_project_20260201_090528
RAW_INSTANCE_DIR: /content/fruitface/instance
INSTANCE_512_DIR: /content/fruitface/instance_512
Saved meta -> /content/drive/MyDrive/fruitface_project_20260201_090528/project_meta.json


In [ ]:
import os
os.environ["OUT_ROOT"] = OUT_ROOT  # теперь $OUT_ROOT виден в ! / %%bash
print("OUT_ROOT env set:", os.environ["OUT_ROOT"])


OUT_ROOT env set: /content/drive/MyDrive/fruitface_project_20260201_090528


In [ ]:
import os
OUT_ROOT = "/content/drive/MyDrive/fruitface_project_20260131_200057"
os.makedirs(OUT_ROOT, exist_ok=True)
os.environ["OUT_ROOT"] = OUT_ROOT
print("OUT_ROOT fixed:", OUT_ROOT)


OUT_ROOT fixed: /content/drive/MyDrive/fruitface_project_20260131_200057


In [ ]:
!ls -la /content/diffusers/examples/textual_inversion | head
!ls -la /content/diffusers/examples/textual_inversion/textual_inversion.py


total 160
drwxr-xr-x  2 root root  4096 Feb  1 08:57 .
drwxr-xr-x 25 root root  4096 Feb  1 08:57 ..
-rw-r--r--  1 root root  5576 Feb  1 08:57 README.md
-rw-r--r--  1 root root  1425 Feb  1 08:57 README_sdxl.md
-rw-r--r--  1 root root    74 Feb  1 08:57 requirements_flax.txt
-rw-r--r--  1 root root    76 Feb  1 08:57 requirements.txt
-rw-r--r--  1 root root  5777 Feb  1 08:57 test_textual_inversion.py
-rw-r--r--  1 root root  5768 Feb  1 08:57 test_textual_inversion_sdxl.py
-rw-r--r--  1 root root 25657 Feb  1 08:57 textual_inversion_flax.py
-rw-r--r-- 1 root root 40577 Feb  1 08:57 /content/diffusers/examples/textual_inversion/textual_inversion.py


In [ ]:
!accelerate config default


accelerate configuration saved at /root/.cache/huggingface/accelerate/default_config.yaml


In [ ]:
import os
from glob import glob
from PIL import Image

os.makedirs(INSTANCE_512_DIR, exist_ok=True)

def center_crop_resize(im, size=512):
    w, h = im.size
    m = min(w, h)
    left = (w - m)//2
    top  = (h - m)//2
    im = im.crop((left, top, left+m, top+m))
    return im.resize((size, size), Image.BICUBIC)

paths = sorted(glob(os.path.join(RAW_INSTANCE_DIR, "*")))
assert len(paths) > 0, f"Папка пустая: {RAW_INSTANCE_DIR}"
print("Found images:", len(paths))

for i, p in enumerate(paths):
    im = Image.open(p).convert("RGB")
    im = center_crop_resize(im, 512)
    im.save(os.path.join(INSTANCE_512_DIR, f"{i:03d}.png"))

print("Prepared:", len(glob(INSTANCE_512_DIR+"/*.png")), "images in", INSTANCE_512_DIR)


Found images: 10
Prepared: 10 images in /content/fruitface/instance_512


In [ ]:
import os, shutil
from glob import glob

SUBSETS_ROOT = "/content/fruitface/subsets"
os.makedirs(SUBSETS_ROOT, exist_ok=True)

all_imgs = sorted(glob(os.path.join(INSTANCE_512_DIR, "*.png")))
assert len(all_imgs) >= 10, "Нужно 10 картинок в instance_512"

subset_sizes = [3, 5, 10]
for n in subset_sizes:
    dst = os.path.join(SUBSETS_ROOT, f"n{n}")
    os.makedirs(dst, exist_ok=True)
    # очищаем папку
    for f in glob(os.path.join(dst, "*.png")):
        os.remove(f)
    # копируем первые n
    for i in range(n):
        shutil.copy(all_imgs[i], os.path.join(dst, f"{i:03d}.png"))
    print("Created subset:", dst, "count =", len(glob(dst+"/*.png")))


Created subset: /content/fruitface/subsets/n3 count = 3
Created subset: /content/fruitface/subsets/n5 count = 5
Created subset: /content/fruitface/subsets/n10 count = 10


In [ ]:
!huggingface-cli login


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: fineGrained).
The token `SDXL_V1` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-crede

In [ ]:
!rm -rf /content/diffusers
!git clone -q https://github.com/huggingface/diffusers /content/diffusers
%cd /content/diffusers
!git checkout -q v0.36.0
%cd /content


/content/diffusers
/content


In [ ]:
%%bash
cd /content/diffusers/examples/textual_inversion

export MODEL_NAME="runwayml/stable-diffusion-v1-5"
export DATA_DIR="/content/fruitface/instance_512"
export OUTPUT_DIR="$OUT_ROOT/TI_base"

accelerate launch --mixed_precision="fp16" textual_inversion.py \
  --pretrained_model_name_or_path="$MODEL_NAME" \
  --train_data_dir="$DATA_DIR" \
  --learnable_property="object" \
  --placeholder_token="<fruitface>" \
  --initializer_token="fruit" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --max_train_steps=1200 \
  --learning_rate=5e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --checkpointing_steps=200 \
  --validation_prompt="a photo of <fruitface> fruit with a cute human face, studio lighting, high detail" \
  --num_validation_images=4 \
  --validation_steps=200 \
  --report_to="tensorboard" \
  --logging_dir="$OUTPUT_DIR/logs" \
  --output_dir="$OUTPUT_DIR"


2026-01-31 20:05:11.899651: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769889911.921196    8817 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769889911.927741    8817 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769889911.944164    8817 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769889911.944188    8817 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769889911.944190    8817 computation_placer.cc:177] computation placer alr

In [ ]:
%%bash
set -e
cd /content/diffusers/examples/dreambooth

export MODEL_NAME="runwayml/stable-diffusion-v1-5"
export INSTANCE_DIR="/content/fruitface/instance_512"
export OUTPUT_DIR="$OUT_ROOT/LORA_base_r8"

accelerate launch --mixed_precision="bf16" train_dreambooth_lora.py \
  --pretrained_model_name_or_path="$MODEL_NAME" \
  --instance_data_dir="$INSTANCE_DIR" \
  --output_dir="$OUTPUT_DIR" \
  --instance_prompt="a photo of sksfruitface fruit with a cute human face" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=1200 \
  --checkpointing_steps=200 \
  --validation_prompt="a photo of sksfruitface fruit with a cute human face, studio lighting, high detail" \
  --validation_epochs=1 \
  --num_validation_images=4 \
  --report_to="tensorboard" \
  --logging_dir="$OUTPUT_DIR/logs" \
  --allow_tf32 \
  --seed=123 \
  --rank=8


2026-01-31 20:17:21.932365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769890641.953481   12410 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769890641.959879   12410 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769890641.975815   12410 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769890641.975858   12410 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769890641.975860   12410 computation_placer.cc:177] computation placer alr

In [ ]:
%%bash
set -e
cd /content/diffusers/examples/dreambooth

export MODEL_NAME="runwayml/stable-diffusion-v1-5"
export INSTANCE_DIR="/content/fruitface/instance_512"
export CLASS_DIR="/content/fruitface/class_images_512"
export OUTPUT_DIR="$OUT_ROOT/DB_full_fast"

accelerate launch --mixed_precision="bf16" train_dreambooth.py \
  --pretrained_model_name_or_path="$MODEL_NAME" \
  --instance_data_dir="$INSTANCE_DIR" \
  --class_data_dir="$CLASS_DIR" \
  --output_dir="$OUTPUT_DIR" \
  --with_prior_preservation \
  --prior_loss_weight=1.0 \
  --instance_prompt="a photo of sksfruitface fruit with a cute human face" \
  --class_prompt="a photo of fruit" \
  --num_class_images=60 \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=2e-6 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=400 \
  --checkpointing_steps=400 \
  --validation_prompt="a photo of sksfruitface fruit with a cute human face, studio lighting, high detail" \
  --num_validation_images=2 \
  --report_to="tensorboard" \
  --logging_dir="$OUTPUT_DIR/logs" \
  --allow_tf32 \
  --seed=123 \
  --use_8bit_adam


2026-02-01 01:10:09.957303: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769908209.978596   86908 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769908209.985207   86908 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769908210.002170   86908 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769908210.002194   86908 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769908210.002196   86908 computation_placer.cc:177] computation placer alr

In [ ]:
%%bash
set -e
cd /content/diffusers/examples/dreambooth

export MODEL_NAME="runwayml/stable-diffusion-v1-5"

for N in 3 5 10
do
  export INSTANCE_DIR="/content/fruitface/subsets/n${N}"
  export OUTPUT_DIR="$OUT_ROOT/EXP1_images_n${N}_r8_fast"

  accelerate launch --mixed_precision="bf16" train_dreambooth_lora.py \
    --pretrained_model_name_or_path="$MODEL_NAME" \
    --instance_data_dir="$INSTANCE_DIR" \
    --output_dir="$OUTPUT_DIR" \
    --instance_prompt="a photo of sksfruitface fruit with a cute human face" \
    --resolution=512 \
    --train_batch_size=1 \
    --gradient_accumulation_steps=1 \
    --learning_rate=1e-4 \
    --lr_scheduler="constant" \
    --lr_warmup_steps=0 \
    --max_train_steps=450 \
    --checkpointing_steps=450 \
    --validation_prompt="a photo of sksfruitface fruit with a cute human face, studio lighting, high detail" \
    --validation_epochs=1 \
    --num_validation_images=2 \
    --report_to="tensorboard" \
    --logging_dir="$OUTPUT_DIR/logs" \
    --allow_tf32 \
    --seed=123 \
    --rank=8
done


In [ ]:
%%bash
set -e
cd /content/diffusers/examples/dreambooth

export MODEL_NAME="runwayml/stable-diffusion-v1-5"
export INSTANCE_DIR="/content/fruitface/instance_512"

for R in 4 16
do
  export OUTPUT_DIR="$OUT_ROOT/EXP2_rank_r${R}_fast"

  accelerate launch --mixed_precision="bf16" train_dreambooth_lora.py \
    --pretrained_model_name_or_path="$MODEL_NAME" \
    --instance_data_dir="$INSTANCE_DIR" \
    --output_dir="$OUTPUT_DIR" \
    --instance_prompt="a photo of sksfruitface fruit with a cute human face" \
    --resolution=512 \
    --train_batch_size=1 \
    --gradient_accumulation_steps=1 \
    --learning_rate=1e-4 \
    --lr_scheduler="constant" \
    --lr_warmup_steps=0 \
    --max_train_steps=450 \
    --checkpointing_steps=450 \
    --validation_prompt="a photo of sksfruitface fruit with a cute human face, studio lighting, high detail" \
    --validation_epochs=1 \
    --num_validation_images=2 \
    --report_to="tensorboard" \
    --logging_dir="$OUTPUT_DIR/logs" \
    --allow_tf32 \
    --seed=123 \
    --rank=$R
done


In [ ]:
%%bash
set -e
cd /content/diffusers/examples/dreambooth

export MODEL_NAME="runwayml/stable-diffusion-v1-5"
export INSTANCE_DIR="/content/fruitface/instance_512"

# OFF
export OUTPUT_DIR="$OUT_ROOT/EXP3_textenc_OFF_fast"
accelerate launch --mixed_precision="bf16" train_dreambooth_lora.py \
  --pretrained_model_name_or_path="$MODEL_NAME" \
  --instance_data_dir="$INSTANCE_DIR" \
  --output_dir="$OUTPUT_DIR" \
  --instance_prompt="a photo of sksfruitface fruit with a cute human face" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=450 \
  --checkpointing_steps=450 \
  --validation_prompt="a photo of sksfruitface fruit with a cute human face, studio lighting, high detail" \
  --validation_epochs=1 \
  --num_validation_images=2 \
  --report_to="tensorboard" \
  --logging_dir="$OUTPUT_DIR/logs" \
  --allow_tf32 \
  --seed=123 \
  --rank=8

# ON
export OUTPUT_DIR="$OUT_ROOT/EXP3_textenc_ON_fast"
accelerate launch --mixed_precision="bf16" train_dreambooth_lora.py \
  --pretrained_model_name_or_path="$MODEL_NAME" \
  --instance_data_dir="$INSTANCE_DIR" \
  --output_dir="$OUTPUT_DIR" \
  --instance_prompt="a photo of sksfruitface fruit with a cute human face" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --text_encoder_lr=5e-5 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=450 \
  --checkpointing_steps=450 \
  --validation_prompt="a photo of sksfruitface fruit with a cute human face, studio lighting, high detail" \
  --validation_epochs=1 \
  --num_validation_images=2 \
  --report_to="tensorboard" \
  --logging_dir="$OUTPUT_DIR/logs" \
  --allow_tf32 \
  --seed=123 \
  --rank=8 \
  --train_text_encoder


In [ ]:
import os, glob, json
from datetime import datetime

def exists_any(patterns):
    for p in patterns:
        if glob.glob(p):
            return True
    return False

def list_some(path, n=25):
    items = sorted(glob.glob(os.path.join(path, "*")))
    return [os.path.basename(x) for x in items[:n]]

def check_run(name, path, must_have_patterns):
    ok_path = os.path.exists(path)
    ok_files = exists_any([os.path.join(path, pat) for pat in must_have_patterns]) if ok_path else False
    return {
        "name": name,
        "path": path,
        "exists": ok_path,
        "has_required_files": ok_files,
        "sample_files": list_some(path) if ok_path else []
    }

# ожидаемые папки (под твой текущий код)
runs_to_check = [
    ("TI_base", os.path.join(OUT_ROOT, "TI_base"), ["learned_embeds*.bin", "learned_embeds*.safetensors", "*.bin", "*.safetensors"]),
    ("LORA_base_r8", os.path.join(OUT_ROOT, "LORA_base_r8"), ["pytorch_lora_weights*.safetensors", "*lora*.safetensors", "*.safetensors"]),
    ("DB_full_fast", os.path.join(OUT_ROOT, "DB_full_fast"), ["model_index.json", "unet/*", "vae/*", "text_encoder/*"]),
    ("EXP1_n3", os.path.join(OUT_ROOT, "EXP1_images_n3_r8_fast"), ["pytorch_lora_weights*.safetensors", "*.safetensors"]),
    ("EXP1_n5", os.path.join(OUT_ROOT, "EXP1_images_n5_r8_fast"), ["pytorch_lora_weights*.safetensors", "*.safetensors"]),
    ("EXP1_n10", os.path.join(OUT_ROOT, "EXP1_images_n10_r8_fast"), ["pytorch_lora_weights*.safetensors", "*.safetensors"]),
    ("EXP2_r4", os.path.join(OUT_ROOT, "EXP2_rank_r4_fast"), ["pytorch_lora_weights*.safetensors", "*.safetensors"]),
    ("EXP2_r16", os.path.join(OUT_ROOT, "EXP2_rank_r16_fast"), ["pytorch_lora_weights*.safetensors", "*.safetensors"]),
    ("EXP3_textenc_OFF", os.path.join(OUT_ROOT, "EXP3_textenc_OFF_fast"), ["pytorch_lora_weights*.safetensors", "*.safetensors"]),
    ("EXP3_textenc_ON", os.path.join(OUT_ROOT, "EXP3_textenc_ON_fast"), ["pytorch_lora_weights*.safetensors", "*.safetensors"]),
]

report = {
    "OUT_ROOT": OUT_ROOT,
    "timestamp": datetime.now().isoformat(),
    "checks": []
}

for name, path, pats in runs_to_check:
    info = check_run(name, path, pats)
    report["checks"].append(info)

# печать для тебя
for c in report["checks"]:
    status = "OK" if (c["exists"] and c["has_required_files"]) else "MISSING"
    print(f"[{status}] {c['name']}: {c['path']}")
    if c["exists"]:
        print("  sample:", c["sample_files"][:8])

# сохраним в файл для рапорта
out_check_path = os.path.join(OUT_ROOT, "artifacts_check.json")
with open(out_check_path, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved check report ->", out_check_path)


In [ ]:
import os, torch
from PIL import Image
from diffusers import StableDiffusionPipeline

device = "cuda"
dtype = torch.float16

RESULTS_DIR = os.path.join(OUT_ROOT, "results_grids")
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_grid(images, path, cols=2):
    w, h = images[0].size
    rows = (len(images) + cols - 1)//cols
    grid = Image.new("RGB", (cols*w, rows*h))
    for i, im in enumerate(images):
        grid.paste(im, ((i % cols)*w, (i // cols)*h))
    grid.save(path)

@torch.inference_mode()
def gen_grid(pipe, prompts, out_path, seed=123, steps=30, cfg=7.0):
    g = torch.Generator(device=device).manual_seed(seed)
    ims = []
    for p in prompts:
        ims.append(pipe(
            p,
            negative_prompt=NEG_PROMPT,
            num_inference_steps=steps,
            guidance_scale=cfg,
            generator=g
        ).images[0])
    save_grid(ims, out_path, cols=2)
    return out_path

def base_pipe():
    return StableDiffusionPipeline.from_pretrained(
        BASE_MODEL, torch_dtype=dtype, safety_checker=None
    ).to(device)

def lora_pipe(lora_dir):
    pipe = base_pipe()
    pipe.load_lora_weights(lora_dir)
    return pipe

def db_pipe(db_dir):
    return StableDiffusionPipeline.from_pretrained(
        db_dir, torch_dtype=dtype, safety_checker=None
    ).to(device)

def ti_pipe(ti_dir, token="<fruitface>"):
    pipe = base_pipe()
    pipe.load_textual_inversion(ti_dir, token=token)
    return pipe

print("Saving grids to:", RESULTS_DIR)

# --- BASE ---
pipe = base_pipe()
gen_grid(pipe, TEST_PROMPTS, os.path.join(RESULTS_DIR, "BASE.png"), seed=GLOBAL_SEED)

# --- Textual Inversion (заменяем token в prompts) ---
pipe = ti_pipe(os.path.join(OUT_ROOT, "TI_base"), token="<fruitface>")
ti_prompts = [p.replace(INSTANCE_TOKEN, "<fruitface>") for p in TEST_PROMPTS]
gen_grid(pipe, ti_prompts, os.path.join(RESULTS_DIR, "TI.png"), seed=GLOBAL_SEED)

# --- LoRA baseline ---
pipe = lora_pipe(os.path.join(OUT_ROOT, "LORA_base_r8"))
gen_grid(pipe, TEST_PROMPTS, os.path.join(RESULTS_DIR, "LORA.png"), seed=GLOBAL_SEED)

# --- DreamBooth full ---
pipe = db_pipe(os.path.join(OUT_ROOT, "DB_full_fast"))
gen_grid(pipe, TEST_PROMPTS, os.path.join(RESULTS_DIR, "DB.png"), seed=GLOBAL_SEED)

# --- EXP1: n=3/5/10 ---
for n in [3,5,10]:
    p = os.path.join(OUT_ROOT, f"EXP1_images_n{n}_r8_fast")
    if os.path.exists(p):
        pipe = lora_pipe(p)
        gen_grid(pipe, TEST_PROMPTS, os.path.join(RESULTS_DIR, f"EXP1_n{n}.png"), seed=GLOBAL_SEED)

# --- EXP2: rank 4/16 ---
for r in [4,16]:
    p = os.path.join(OUT_ROOT, f"EXP2_rank_r{r}_fast")
    if os.path.exists(p):
        pipe = lora_pipe(p)
        gen_grid(pipe, TEST_PROMPTS, os.path.join(RESULTS_DIR, f"EXP2_r{r}.png"), seed=GLOBAL_SEED)

# --- EXP3: text encoder OFF/ON ---
for tag in ["OFF", "ON"]:
    p = os.path.join(OUT_ROOT, f"EXP3_textenc_{tag}_fast")
    if os.path.exists(p):
        pipe = lora_pipe(p)
        gen_grid(pipe, TEST_PROMPTS, os.path.join(RESULTS_DIR, f"EXP3_{tag}.png"), seed=GLOBAL_SEED)

print("Done. Files:")
print(sorted([os.path.basename(x) for x in os.listdir(RESULTS_DIR)]))


In [ ]:
%load_ext tensorboard
%tensorboard --logdir OUT_ROOT


In [ ]:
import shutil
zip_path = shutil.make_archive(OUT_ROOT, "zip", OUT_ROOT)
print("ZIP:", zip_path)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
!pip -q install -U pip

!pip -q install \
  diffusers==0.36.0 \
  transformers==4.57.3 \
  accelerate==1.12.0 \
  peft==0.18.1 \
  bitsandbytes==0.49.1 \
  safetensors \
  datasets \
  ftfy \
  tensorboard \
  sentencepiece \
  protobuf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.4 MB/s eta 0:00:00


In [ ]:
!rm -rf /content/diffusers
!git clone -q https://github.com/huggingface/diffusers /content/diffusers
%cd /content/diffusers
!git checkout -q v0.36.0
%cd /content
!accelerate config default


/content/diffusers
/content
accelerate configuration saved at /root/.cache/huggingface/accelerate/default_config.yaml


In [ ]:
import os, glob

cands = sorted(glob.glob("/content/drive/MyDrive/fruitface_project_*"), key=os.path.getmtime)
assert len(cands) > 0, "Не нашёл папки fruitface_project_* в MyDrive"
OUT_ROOT = cands[-1]  # самая свежая
os.environ["OUT_ROOT"] = OUT_ROOT
print("Using OUT_ROOT:", OUT_ROOT)

# посмотрим что там есть
!ls -la "$OUT_ROOT"


Using OUT_ROOT: /content/drive/MyDrive/fruitface_project_20260131_200057
total 61
drwx------ 2 root root 4096 Jan 31 21:00 DB_full
drwx------ 2 root root 4096 Feb  1 01:10 DB_full_fast
drwx------ 2 root root 4096 Jan 31 23:37 EXP1_images_n10_r8
drwx------ 2 root root 4096 Feb  1 01:58 EXP1_images_n10_r8_fast
drwx------ 2 root root 4096 Jan 31 20:15 EXP1_images_n3_r8
drwx------ 2 root root 4096 Feb  1 01:13 EXP1_images_n3_r8_fast
drwx------ 2 root root 4096 Jan 31 22:40 EXP1_images_n5_r8
drwx------ 2 root root 4096 Feb  1 01:41 EXP1_images_n5_r8_fast
drwx------ 2 root root 4096 Feb  1 00:40 EXP2_rank_r16
drwx------ 2 root root 4096 Feb  1 02:20 EXP2_rank_r16_fast
drwx------ 2 root root 4096 Feb  1 00:08 EXP2_rank_r4
drwx------ 2 root root 4096 Feb  1 02:09 EXP2_rank_r4_fast
drwx------ 2 root root 4096 Feb  1 02:30 EXP3_textenc_OFF_fast
drwx------ 2 root root 4096 Jan 31 20:14 LORA_base_r8
-rw------- 1 root root  243 Jan 31 20:00 project_meta.json
drwx------ 2 root root 4096 Jan 31 20:05

In [ ]:
import os, glob

def has_lora_weights(d):
    return any(glob.glob(os.path.join(d, "pytorch_lora_weights*.safetensors"))) or any(glob.glob(os.path.join(d, "*lora*.safetensors")))

def has_db_model(d):
    return os.path.exists(os.path.join(d, "model_index.json"))

def has_ti_embed(d):
    return any(glob.glob(os.path.join(d, "learned_embeds*.bin"))) or any(glob.glob(os.path.join(d, "learned_embeds*.safetensors")))

checks = {
    "TI_base": has_ti_embed(os.path.join(OUT_ROOT, "TI_base")),
    "LORA_base_r8": has_lora_weights(os.path.join(OUT_ROOT, "LORA_base_r8")),
    "DB_full_fast": has_db_model(os.path.join(OUT_ROOT, "DB_full_fast")),
    "EXP1_n3": has_lora_weights(os.path.join(OUT_ROOT, "EXP1_images_n3_r8_fast")),
    "EXP1_n5": has_lora_weights(os.path.join(OUT_ROOT, "EXP1_images_n5_r8_fast")),
    "EXP1_n10": has_lora_weights(os.path.join(OUT_ROOT, "EXP1_images_n10_r8_fast")),
    "EXP2_r4": has_lora_weights(os.path.join(OUT_ROOT, "EXP2_rank_r4_fast")),
    "EXP2_r16": has_lora_weights(os.path.join(OUT_ROOT, "EXP2_rank_r16_fast")),
    "EXP3_OFF": has_lora_weights(os.path.join(OUT_ROOT, "EXP3_textenc_OFF_fast")),
    "EXP3_ON": has_lora_weights(os.path.join(OUT_ROOT, "EXP3_textenc_ON_fast")),
}

for k,v in checks.items():
    print(f"{k:12s} ->", "OK" if v else "MISSING")


TI_base      -> OK
LORA_base_r8 -> OK
DB_full_fast -> OK
EXP1_n3      -> OK
EXP1_n5      -> OK
EXP1_n10     -> OK
EXP2_r4      -> OK
EXP2_r16     -> OK
EXP3_OFF     -> OK
EXP3_ON      -> MISSING


In [ ]:
import os, glob

p = os.path.join(OUT_ROOT, "EXP3_textenc_ON_fast")
print("Path:", p, "exists:", os.path.exists(p))

if os.path.exists(p):
    print("\nTop-level files:")
    for f in sorted(glob.glob(p+"/*"))[:50]:
        print(" ", os.path.basename(f))

    print("\nCheckpoints:")
    for f in sorted(glob.glob(p+"/checkpoint-*"))[:20]:
        print(" ", os.path.basename(f))

    print("\nLoRA weights candidates:")
    for f in glob.glob(p+"/**/*.safetensors", recursive=True)[:50]:
        print(" ", f.replace(p+"/",""))


Path: /content/drive/MyDrive/fruitface_project_20260131_200057/EXP3_textenc_ON_fast exists: False


In [ ]:
import glob
print("\n".join(sorted(glob.glob(OUT_ROOT + "/EXP3*"))))


/content/drive/MyDrive/fruitface_project_20260131_200057/EXP3_textenc_OFF_fast


In [ ]:
%%bash
set -e
cd /content/diffusers/examples/dreambooth

export MODEL_NAME="runwayml/stable-diffusion-v1-5"
export INSTANCE_DIR="/content/fruitface/instance_512"
export OUTPUT_DIR="$OUT_ROOT/EXP3_textenc_ON_fast"

mkdir -p "$OUTPUT_DIR"

# ON: train text encoder (без --text_encoder_lr, поэтому делаем общий LR ниже)
accelerate launch --mixed_precision="bf16" train_dreambooth_lora.py \
  --pretrained_model_name_or_path="$MODEL_NAME" \
  --instance_data_dir="$INSTANCE_DIR" \
  --output_dir="$OUTPUT_DIR" \
  --instance_prompt="a photo of sksfruitface fruit with a cute human face" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=5e-5 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=450 \
  --checkpointing_steps=450 \
  --validation_prompt="a photo of sksfruitface fruit with a cute human face, studio lighting, high detail" \
  --validation_epochs=1 \
  --num_validation_images=2 \
  --report_to="tensorboard" \
  --logging_dir="$OUTPUT_DIR/logs" \
  --allow_tf32 \
  --seed=123 \
  --rank=8 \
  --train_text_encoder


2026-02-01 09:14:25.296454: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-01 09:14:25.313755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769937265.334894    6471 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769937265.341433    6471 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769937265.357574    6471 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
import os, glob

p = os.path.join(OUT_ROOT, "EXP3_textenc_ON_fast")
print("OUT:", p)
print("exists:", os.path.exists(p))

if os.path.exists(p):
    print("\nTop-level:")
    for f in sorted(glob.glob(p+"/*"))[:50]:
        print(" ", os.path.basename(f))

    print("\nCheckpoints:")
    for f in sorted(glob.glob(p+"/checkpoint-*"))[:20]:
        print(" ", os.path.basename(f))

    print("\nLoRA candidates (.safetensors/.bin):")
    cand = glob.glob(p+"/**/*.safetensors", recursive=True) + glob.glob(p+"/**/*.bin", recursive=True)
    for f in sorted(cand)[:50]:
        print(" ", f.replace(p+"/",""))


OUT: /content/drive/MyDrive/fruitface_project_20260131_200057/EXP3_textenc_ON_fast
exists: True

Top-level:
  checkpoint-450
  logs
  pytorch_lora_weights.safetensors

Checkpoints:
  checkpoint-450

LoRA candidates (.safetensors/.bin):
  checkpoint-450/optimizer.bin
  checkpoint-450/pytorch_lora_weights.safetensors
  checkpoint-450/scheduler.bin
  pytorch_lora_weights.safetensors


In [ ]:
import os, torch
from diffusers import StableDiffusionPipeline
from safetensors.torch import load_file

BASE_MODEL = "runwayml/stable-diffusion-v1-5"

prompts = [
    "a photo of sksfruitface fruit with a cute human face, studio lighting, high detail",
    "sksfruitface banana with a cute human face, macro photo, shallow depth of field",
    "sksfruitface apple with a cute human face, cinematic lighting, 50mm",
]
neg = "blurry, low quality, worst quality, deformed, bad anatomy"
seed = 123

out_dir = os.path.join(OUT_ROOT, "report_samples")
os.makedirs(out_dir, exist_ok=True)

def find_lora_file(exp_dir):
    # ищем файл лоры (берём последний по имени/пути — обычно checkpoint-450)
    cand = sorted(
        glob.glob(os.path.join(exp_dir, "**/*.safetensors"), recursive=True) +
        glob.glob(os.path.join(exp_dir, "**/*.bin"), recursive=True)
    )
    # часто лежит в checkpoint-*/pytorch_lora_weights.safetensors
    # поэтому проще взять самый глубокий/последний
    return cand[-1] if cand else None

def load_pipe():
    pipe = StableDiffusionPipeline.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        safety_checker=None,
    )
    pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")
    pipe.enable_attention_slicing()
    return pipe

def apply_lora(pipe, lora_path):
    # diffusers умеет load_lora_weights, но формат бывает разный.
    # попробуем нативно:
    try:
        pipe.load_lora_weights(os.path.dirname(lora_path), weight_name=os.path.basename(lora_path))
        return "pipe.load_lora_weights"
    except Exception as e:
        return f"FAILED load_lora_weights: {e}"

import glob

exp_off = os.path.join(OUT_ROOT, "EXP3_textenc_OFF_fast")
exp_on  = os.path.join(OUT_ROOT, "EXP3_textenc_ON_fast")
lora_off = find_lora_file(exp_off)
lora_on  = find_lora_file(exp_on)

print("LoRA OFF:", lora_off)
print("LoRA ON :", lora_on)

pipe = load_pipe()

def gen_set(tag, lora_path):
    # сбрасываем лору (если до этого грузили)
    try:
        pipe.unload_lora_weights()
    except:
        pass

    if lora_path:
        print(tag, apply_lora(pipe, lora_path))
    else:
        print(tag, "NO LORA FILE FOUND -> will generate base")

    g = torch.Generator(device=pipe.device).manual_seed(seed)
    for i, pr in enumerate(prompts):
        img = pipe(
            pr, negative_prompt=neg,
            num_inference_steps=30,
            guidance_scale=7.0,
            generator=g
        ).images[0]
        img.save(os.path.join(out_dir, f"{tag}_{i:02d}.png"))

gen_set("OFF", lora_off)
gen_set("ON",  lora_on)

print("Saved to:", out_dir)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


LoRA OFF: /content/drive/MyDrive/fruitface_project_20260131_200057/EXP3_textenc_OFF_fast/pytorch_lora_weights.safetensors
LoRA ON : /content/drive/MyDrive/fruitface_project_20260131_200057/EXP3_textenc_ON_fast/pytorch_lora_weights.safetensors


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected:

OFF pipe.load_lora_weights


... (4 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/fruitface_project_20260131_200057/report_samples


In [ ]:
import os, re, glob

DRIVE_ROOT = "/content/drive/MyDrive"
cands = sorted(glob.glob(os.path.join(DRIVE_ROOT, "fruitface_project_*")), key=os.path.getmtime, reverse=True)

assert len(cands) > 0, "Не нашёл fruitface_project_* в MyDrive"
OUT_ROOT = cands[0]   # самый новый
os.environ["OUT_ROOT"] = OUT_ROOT

print("Using OUT_ROOT =", OUT_ROOT)
print("All candidates:")
for p in cands[:5]:
    print("  ", p)


Using OUT_ROOT = /content/drive/MyDrive/fruitface_project_20260131_200057
All candidates:
   /content/drive/MyDrive/fruitface_project_20260131_200057
   /content/drive/MyDrive/fruitface_project_20260201_090528
   /content/drive/MyDrive/fruitface_project_20260131_195919


In [ ]:
import os, glob
from PIL import Image

RAW_INSTANCE_DIR = "/content/fruitface/instance"
INSTANCE_512_DIR = "/content/fruitface/instance_512"

def check_dir_images(dir_path, exts=("png","jpg","jpeg","webp")):
    files = []
    for e in exts:
        files += glob.glob(os.path.join(dir_path, f"*.{e}"))
        files += glob.glob(os.path.join(dir_path, f"*.{e.upper()}"))
    return sorted(files)

raw = check_dir_images(RAW_INSTANCE_DIR)
img512 = check_dir_images(INSTANCE_512_DIR)

print("RAW_INSTANCE_DIR:", RAW_INSTANCE_DIR, "count =", len(raw))
print("INSTANCE_512_DIR:", INSTANCE_512_DIR, "count =", len(img512))

assert len(raw) >= 10, "В RAW_INSTANCE_DIR должно быть >= 10 исходных картинок"
assert len(img512) >= 10, "В INSTANCE_512_DIR должно быть >= 10 картинок (512×512)"

# проверим первые 10 на размер/мод
bad = []
for p in img512[:10]:
    im = Image.open(p)
    if im.size != (512,512):
        bad.append((os.path.basename(p), im.size))
    if im.mode != "RGB":
        # не критично, но лучше RGB
        pass

if bad:
    print("BAD SIZES:", bad)
    raise AssertionError("Есть картинки не 512×512 — пересоздай instance_512")

print("OK: первые 10 картинок — 512×512")
print("Example files:", [os.path.basename(x) for x in img512[:10]])


RAW_INSTANCE_DIR: /content/fruitface/instance count = 10
INSTANCE_512_DIR: /content/fruitface/instance_512 count = 10
OK: первые 10 картинок — 512×512
Example files: ['000.png', '001.png', '002.png', '003.png', '004.png', '005.png', '006.png', '007.png', '008.png', '009.png']


In [ ]:
import os, glob, json

def exists_any(patterns):
    for pat in patterns:
        if glob.glob(pat, recursive=True):
            return True
    return False

def find_latest_checkpoint(dir_path):
    cps = sorted(glob.glob(os.path.join(dir_path, "checkpoint-*")))
    if not cps:
        return None
    # checkpoint-450 -> 450
    def step(p):
        try:
            return int(os.path.basename(p).split("-")[1])
        except:
            return -1
    cps = sorted(cps, key=step)
    return cps[-1]

def has_ti(dir_path):
    return exists_any([
        os.path.join(dir_path, "learned_embeds.safetensors"),
        os.path.join(dir_path, "learned_embeds-steps-*.safetensors"),
        os.path.join(dir_path, "checkpoint-*", "learned_embeds*.safetensors"),
        os.path.join(dir_path, "checkpoint-*", "learned_embeds*.bin"),
    ])

def has_db_full(dir_path):
    return os.path.exists(os.path.join(dir_path, "model_index.json"))

def has_lora(dir_path):
    return exists_any([
        os.path.join(dir_path, "pytorch_lora_weights.safetensors"),
        os.path.join(dir_path, "checkpoint-*", "pytorch_lora_weights.safetensors"),
        os.path.join(dir_path, "**", "pytorch_lora_weights.safetensors"),
    ])

expected = [
    ("TI_base",        "TI"),
    ("LORA_base_r8",   "LoRA"),
    ("DB_full_fast",   "DBFULL"),
    ("EXP1_images_n3_r8_fast",  "LoRA"),
    ("EXP1_images_n5_r8_fast",  "LoRA"),
    ("EXP1_images_n10_r8_fast", "LoRA"),
    ("EXP2_rank_r4_fast",       "LoRA"),
    ("EXP2_rank_r16_fast",      "LoRA"),
    ("EXP3_textenc_OFF_fast",   "LoRA"),
    ("EXP3_textenc_ON_fast",    "LoRA"),
]

report = {"OUT_ROOT": OUT_ROOT, "items": []}

print("=== CHECK OUT_ROOT CONTENTS ===")
for name, typ in expected:
    p = os.path.join(OUT_ROOT, name)
    ok = False
    details = {"name": name, "type": typ, "path": p, "exists": os.path.exists(p)}
    if os.path.exists(p):
        if typ == "TI":
            ok = has_ti(p)
        elif typ == "DBFULL":
            ok = has_db_full(p)
        else:
            ok = has_lora(p)

        details["latest_checkpoint"] = find_latest_checkpoint(p)
        details["top_files"] = sorted([os.path.basename(x) for x in glob.glob(p+"/*")])[:50]
    details["ok"] = bool(ok)
    report["items"].append(details)
    print(f"{name:24s} -> {'OK' if ok else 'MISSING'}")

# сохраним audit
audit_path = os.path.join(OUT_ROOT, "report_audit.json")
with open(audit_path, "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", audit_path)


=== CHECK OUT_ROOT CONTENTS ===
TI_base                  -> OK
LORA_base_r8             -> OK
DB_full_fast             -> OK
EXP1_images_n3_r8_fast   -> OK
EXP1_images_n5_r8_fast   -> OK
EXP1_images_n10_r8_fast  -> OK
EXP2_rank_r4_fast        -> OK
EXP2_rank_r16_fast       -> OK
EXP3_textenc_OFF_fast    -> OK
EXP3_textenc_ON_fast     -> OK

Saved: /content/drive/MyDrive/fruitface_project_20260131_200057/report_audit.json


In [ ]:
import os, glob, json, math
import torch
from PIL import Image
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

BASE_MODEL = "runwayml/stable-diffusion-v1-5"

INSTANCE_TOKEN = "sksfruitface"
CLASS_WORD = "fruit"

# твои промпты (как ты писал)
TEST_PROMPTS_LORA_DB = [
    f"a photo of {INSTANCE_TOKEN} fruit with a cute human face, studio lighting, high detail",
    f"{INSTANCE_TOKEN} banana with a cute human face, macro photo, shallow depth of field",
    f"{INSTANCE_TOKEN} apple with a cute human face, cinematic lighting, 50mm",
    f"a bowl of {INSTANCE_TOKEN} fruits with cute human faces, professional food photography",
    f"{INSTANCE_TOKEN} orange with a cute human face, white background product photo",
]
TEST_PROMPTS_TI = [
    "a photo of <fruitface> fruit with a cute human face, studio lighting, high detail",
    "<fruitface> banana with a cute human face, macro photo, shallow depth of field",
    "<fruitface> apple with a cute human face, cinematic lighting, 50mm",
    "a bowl of <fruitface> fruits with cute human faces, professional food photography",
    "<fruitface> orange with a cute human face, white background product photo",
]

NEG_PROMPT = "blurry, low quality, worst quality, deformed, bad anatomy"
GLOBAL_SEED = 123

REPORT_IMG_DIR  = os.path.join(OUT_ROOT, "report_images")
REPORT_GRID_DIR = os.path.join(OUT_ROOT, "report_grids")
os.makedirs(REPORT_IMG_DIR, exist_ok=True)
os.makedirs(REPORT_GRID_DIR, exist_ok=True)

def make_pipe_from_base():
    pipe = StableDiffusionPipeline.from_pretrained(
        BASE_MODEL,
        torch_dtype=DTYPE,
        safety_checker=None,   # быстрее и без “black image”
        requires_safety_checker=False
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(DEVICE)
    pipe.enable_attention_slicing()
    if DEVICE == "cuda":
        pipe.enable_vae_slicing()
    return pipe

def make_pipe_from_db(db_dir):
    pipe = StableDiffusionPipeline.from_pretrained(
        db_dir,
        torch_dtype=DTYPE,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(DEVICE)
    pipe.enable_attention_slicing()
    if DEVICE == "cuda":
        pipe.enable_vae_slicing()
    return pipe

def find_lora_weight(dir_path):
    # приоритет: root/pytorch_lora_weights.safetensors -> latest checkpoint -> любой найденный
    root_w = os.path.join(dir_path, "pytorch_lora_weights.safetensors")
    if os.path.exists(root_w):
        return root_w
    cps = sorted(glob.glob(os.path.join(dir_path, "checkpoint-*")))
    def step(p):
        try: return int(os.path.basename(p).split("-")[1])
        except: return -1
    cps = sorted(cps, key=step, reverse=True)
    for cp in cps:
        w = os.path.join(cp, "pytorch_lora_weights.safetensors")
        if os.path.exists(w):
            return w
    any_w = glob.glob(os.path.join(dir_path, "**", "pytorch_lora_weights.safetensors"), recursive=True)
    return any_w[0] if any_w else None

def find_ti_weight(ti_dir):
    # приоритет: learned_embeds.safetensors -> max steps -> checkpoint
    p0 = os.path.join(ti_dir, "learned_embeds.safetensors")
    if os.path.exists(p0):
        return p0
    cands = glob.glob(os.path.join(ti_dir, "learned_embeds-steps-*.safetensors"))
    if cands:
        def step(p):
            try: return int(os.path.basename(p).split("-steps-")[1].split(".")[0])
            except: return -1
        return sorted(cands, key=step)[-1]
    cands = glob.glob(os.path.join(ti_dir, "checkpoint-*", "learned_embeds*.safetensors"))
    return cands[0] if cands else None

def render(pipe, prompt, out_path, seed=GLOBAL_SEED, steps=30, guidance=7.0):
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    img = pipe(
        prompt=prompt,
        negative_prompt=NEG_PROMPT,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=g
    ).images[0]
    img.save(out_path)
    return out_path

def make_grid(img_paths, cols=5):
    imgs = [Image.open(p).convert("RGB") for p in img_paths]
    if not imgs:
        return None
    w, h = imgs[0].size
    rows = math.ceil(len(imgs)/cols)
    grid = Image.new("RGB", (cols*w, rows*h), (0,0,0))
    for i, im in enumerate(imgs):
        r, c = divmod(i, cols)
        grid.paste(im, (c*w, r*h))
    return grid

def run_experiment(exp_name, kind, path, prompts, steps=30, guidance=7.0):
    out_dir = os.path.join(REPORT_IMG_DIR, exp_name)
    os.makedirs(out_dir, exist_ok=True)

    meta = {"exp": exp_name, "kind": kind, "path": path, "prompts": prompts, "images": []}

    if kind == "BASE":
        pipe = make_pipe_from_base()

    elif kind == "TI":
        pipe = make_pipe_from_base()
        ti_w = find_ti_weight(path)
        assert ti_w is not None, f"Не нашёл TI веса в {path}"
        pipe.load_textual_inversion(ti_w, token="<fruitface>")
        meta["ti_weight"] = ti_w

    elif kind == "LORA":
        pipe = make_pipe_from_base()
        lora_w = find_lora_weight(path)
        assert lora_w is not None, f"Не нашёл LoRA веса в {path}"
        # load_lora_weights принимает либо dir+weight_name, либо сразу путь
        pipe.load_lora_weights(os.path.dirname(lora_w), weight_name=os.path.basename(lora_w))
        pipe.fuse_lora(lora_scale=1.0)
        meta["lora_weight"] = lora_w

    elif kind == "DBFULL":
        pipe = make_pipe_from_db(path)

    else:
        raise ValueError(kind)

    img_paths = []
    for i, pr in enumerate(prompts):
        out_path = os.path.join(out_dir, f"{i:02d}.png")
        render(pipe, pr, out_path, seed=GLOBAL_SEED + i*17, steps=steps, guidance=guidance)
        img_paths.append(out_path)
        meta["images"].append(out_path)

    grid = make_grid(img_paths, cols=len(prompts))
    grid_path = os.path.join(REPORT_GRID_DIR, f"{exp_name}__grid.png")
    if grid:
        grid.save(grid_path)
        meta["grid"] = grid_path

    # чистим чтобы VRAM не копилась
    del pipe
    torch.cuda.empty_cache()

    return meta

# ----------------- что именно генерим -----------------
experiments = []

# baseline (без обучения) для сравнения — тот же смысл, но без токена
experiments.append(("BASELINE", "BASE", None, [
    "a photo of fruit with a cute human face, studio lighting, high detail",
    "banana with a cute human face, macro photo, shallow depth of field",
    "apple with a cute human face, cinematic lighting, 50mm",
    "a bowl of fruits with cute human faces, professional food photography",
    "orange with a cute human face, white background product photo",
]))

# TI
ti_dir = os.path.join(OUT_ROOT, "TI_base")
if os.path.exists(ti_dir):
    experiments.append(("TI_base", "TI", ti_dir, TEST_PROMPTS_TI))

# LoRA base
lora_base = os.path.join(OUT_ROOT, "LORA_base_r8")
if os.path.exists(lora_base):
    experiments.append(("LORA_base_r8", "LORA", lora_base, TEST_PROMPTS_LORA_DB))

# DB full
db_dir = os.path.join(OUT_ROOT, "DB_full_fast")
if os.path.exists(db_dir):
    experiments.append(("DB_full_fast", "DBFULL", db_dir, TEST_PROMPTS_LORA_DB))

# EXP1 subsets
for n in [3,5,10]:
    p = os.path.join(OUT_ROOT, f"EXP1_images_n{n}_r8_fast")
    if os.path.exists(p):
        experiments.append((f"EXP1_n{n}_r8", "LORA", p, TEST_PROMPTS_LORA_DB))

# EXP2 ranks
for r in [4,16]:
    p = os.path.join(OUT_ROOT, f"EXP2_rank_r{r}_fast")
    if os.path.exists(p):
        experiments.append((f"EXP2_rank_r{r}", "LORA", p, TEST_PROMPTS_LORA_DB))

# EXP3 textenc
p_off = os.path.join(OUT_ROOT, "EXP3_textenc_OFF_fast")
if os.path.exists(p_off):
    experiments.append(("EXP3_textenc_OFF", "LORA", p_off, TEST_PROMPTS_LORA_DB))

p_on = os.path.join(OUT_ROOT, "EXP3_textenc_ON_fast")
if os.path.exists(p_on):
    experiments.append(("EXP3_textenc_ON", "LORA", p_on, TEST_PROMPTS_LORA_DB))

print("Will generate for:", [x[0] for x in experiments])


Will generate for: ['BASELINE', 'TI_base', 'LORA_base_r8', 'DB_full_fast', 'EXP1_n3_r8', 'EXP1_n5_r8', 'EXP1_n10_r8', 'EXP2_rank_r4', 'EXP2_rank_r16', 'EXP3_textenc_OFF', 'EXP3_textenc_ON']


In [ ]:
all_meta = []
for exp_name, kind, path, prompts in experiments:
    meta = run_experiment(
        exp_name=exp_name,
        kind=kind,
        path=path,
        prompts=prompts,
        steps=30,       # быстро
        guidance=7.0
    )
    all_meta.append(meta)
    print("DONE:", exp_name, "->", meta.get("grid", ""))

summary_path = os.path.join(OUT_ROOT, "report_summary.json")
with open(summary_path, "w") as f:
    json.dump(all_meta, f, indent=2)

print("Saved summary:", summary_path)
print("Images dir:", REPORT_IMG_DIR)
print("Grids dir:", REPORT_GRID_DIR)


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

... (77 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

DONE: EXP3_textenc_ON -> /content/drive/MyDrive/fruitface_project_20260131_200057/report_grids/EXP3_textenc_ON__grid.png
Saved summary: /content/drive/MyDrive/fruitface_project_20260131_200057/report_summary.json
Images dir: /content/drive/MyDrive/fruitface_project_20260131_200057/report_images
Grids dir: /content/drive/MyDrive/fruitface_project_20260131_200057/report_grids


In [ ]:
import os, glob

print("OUT_ROOT =", OUT_ROOT)

imgs = glob.glob(os.path.join(OUT_ROOT, "report_images", "**", "*.png"), recursive=True)
grids = glob.glob(os.path.join(OUT_ROOT, "report_grids", "*.png"))

print("Generated images:", len(imgs))
print("Generated grids:", len(grids))

assert len(grids) > 0, "Гридов нет — генерация не создала результат"
print("Example grids:")
for p in sorted(grids)[:10]:
    print(" ", os.path.basename(p))

print("OK ✅")


OUT_ROOT = /content/drive/MyDrive/fruitface_project_20260131_200057
Generated images: 55
Generated grids: 11
Example grids:
  BASELINE__grid.png
  DB_full_fast__grid.png
  EXP1_n10_r8__grid.png
  EXP1_n3_r8__grid.png
  EXP1_n5_r8__grid.png
  EXP2_rank_r16__grid.png
  EXP2_rank_r4__grid.png
  EXP3_textenc_OFF__grid.png
  EXP3_textenc_ON__grid.png
  LORA_base_r8__grid.png
OK ✅


In [ ]:
import os, math, torch
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler
from PIL import Image

# ====== НАСТРОЙ ПАПОК ======
OUT_ROOT = "/content/drive/MyDrive/fruitface_project_20260131_200057"  # <-- твой проект
DB_DIR   = os.path.join(OUT_ROOT, "DB_full_fast")
SAVE_DIR = os.path.join(OUT_ROOT, "rerun_DB_full_no_xformers")
os.makedirs(SAVE_DIR, exist_ok=True)

device = "cuda"
dtype = torch.float16

# ====== ГРУЗИМ DB МОДЕЛЬ ======
pipe = StableDiffusionPipeline.from_pretrained(DB_DIR, torch_dtype=dtype, safety_checker=None)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

INSTANCE_TOKEN = "sksfruitface"

PROMPTS = [
    f"photorealistic studio photo of a {INSTANCE_TOKEN} strawberry with a realistic human face embedded in the strawberry surface, centered, sharp focus, high detail",
    f"macro photo of a {INSTANCE_TOKEN} strawberry with a realistic human face seamlessly blended into the fruit, shallow depth of field, high detail",
    f"product photo: a {INSTANCE_TOKEN} strawberry with a realistic human face, white background, sharp focus, high detail",
]

NEG = (
    "cartoon, anime, illustration, painting, drawing, 3d render, CGI, lowres, blurry, noisy, "
    "deformed, disfigured, extra face, extra eyes, face next to fruit, floating head, "
    "text, watermark, logo, worst quality, low quality"
)

num_steps = 60
guidance  = 8.5
n_per_prompt = 4
base_seed = 123

def save_grid(images, cols, path):
    w, h = images[0].size
    rows = math.ceil(len(images) / cols)
    grid = Image.new("RGB", (cols*w, rows*h))
    for i, im in enumerate(images):
        grid.paste(im, ((i % cols)*w, (i // cols)*h))
    grid.save(path)

for pi, prompt in enumerate(PROMPTS):
    imgs = []
    for k in range(n_per_prompt):
        seed = base_seed + 1000*pi + k
        gen = torch.Generator(device=device).manual_seed(seed)
        img = pipe(
            prompt=prompt,
            negative_prompt=NEG,
            num_inference_steps=num_steps,
            guidance_scale=guidance,
            generator=gen
        ).images[0]
        imgs.append(img)
        img.save(os.path.join(SAVE_DIR, f"p{pi:02d}_k{k:02d}_seed{seed}.png"))
    save_grid(imgs, cols=2, path=os.path.join(SAVE_DIR, f"GRID_p{pi:02d}.png"))

print("DONE. Saved to:", SAVE_DIR)


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

... (5 outputs left out)

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

DONE. Saved to: /content/drive/MyDrive/fruitface_project_20260131_200057/rerun_DB_full_no_xformers


In [ ]:
import os, math, torch
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler
from PIL import Image

OUT_ROOT = "/content/drive/MyDrive/fruitface_project_20260131_200057"  # <-- твой проект
TI_DIR   = os.path.join(OUT_ROOT, "TI_base")
TI_EMBED = os.path.join(TI_DIR, "learned_embeds.safetensors")

SAVE_DIR = os.path.join(OUT_ROOT, "rerun_TI_no_xformers")
os.makedirs(SAVE_DIR, exist_ok=True)

BASE_MODEL = "runwayml/stable-diffusion-v1-5"
device = "cuda"
dtype = torch.float16

pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL, torch_dtype=dtype, safety_checker=None)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

TOKEN = "<fruitface>"   # должен совпадать с placeholder_token при обучении
pipe.load_textual_inversion(TI_EMBED, token=TOKEN)

PROMPTS = [
    f"photorealistic studio photo of a {TOKEN} strawberry with a realistic human face embedded in the strawberry surface, centered, sharp focus, high detail",
    f"macro photo of a {TOKEN} strawberry with a realistic human face seamlessly blended into the fruit, shallow depth of field, high detail",
    f"product photo: a {TOKEN} strawberry with a realistic human face, white background, sharp focus, high detail",
]

NEG = (
    "cartoon, anime, illustration, painting, drawing, 3d render, CGI, lowres, blurry, noisy, "
    "deformed, disfigured, extra face, extra eyes, face next to fruit, floating head, "
    "text, watermark, logo, worst quality, low quality"
)

num_steps = 70
guidance  = 9.0
n_per_prompt = 4
base_seed = 123

def save_grid(images, cols, path):
    w, h = images[0].size
    rows = math.ceil(len(images) / cols)
    grid = Image.new("RGB", (cols*w, rows*h))
    for i, im in enumerate(images):
        grid.paste(im, ((i % cols)*w, (i // cols)*h))
    grid.save(path)

for pi, prompt in enumerate(PROMPTS):
    imgs = []
    for k in range(n_per_prompt):
        seed = base_seed + 1000*pi + k
        gen = torch.Generator(device=device).manual_seed(seed)
        img = pipe(
            prompt=prompt,
            negative_prompt=NEG,
            num_inference_steps=num_steps,
            guidance_scale=guidance,
            generator=gen
        ).images[0]
        imgs.append(img)
        img.save(os.path.join(SAVE_DIR, f"p{pi:02d}_k{k:02d}_seed{seed}.png"))
    save_grid(imgs, cols=2, path=os.path.join(SAVE_DIR, f"GRID_p{pi:02d}.png"))

print("DONE. Saved to:", SAVE_DIR)


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

... (5 outputs left out)

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

DONE. Saved to: /content/drive/MyDrive/fruitface_project_20260131_200057/rerun_TI_no_xformers
